In [1]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from docopt import docopt
import community as community
import random
import yaml
from tqdm import tqdm, trange

import time
import pickle

from networkx.algorithms.community.label_propagation import asyn_lpa_communities
from infomap import Infomap


import itertools

from scipy.stats import entropy
import math
import pandas as pd


# Execução de algoritmos de detecção de comunidades

In [2]:
def calc_infomap(G, n_nodes):
    print('Calculating partitions')
    partitions = []
    seeds = []
    for k in trange(1000):
        # Código original
        # seed = int(time.time() * 1000)

        # Correção para garantir que o seed seja único em cada iteração
        seed = int(time.time() * 1000000) + os.getpid() * 1000 + k
        
        seeds.append(seed)
        im = Infomap('-s {0}'.format(seed))
        im.add_nodes(range(n_nodes))
        im.add_links(list(G.edges()))
        im.run('--silent')
        partition = im.get_modules()
        partition_list = []
        for _, comm_index in partition.items():
            partition_list.append(comm_index)
        partitions.append(partition_list)
    partitions = np.array(partitions)
    return partitions, seeds

In [ ]:
def calc_lpa(G, n_nodes):
    print('Calculating partitions')
    partitions = []
    seeds = []
    for k in trange(1000):
        seed = int(time.time() * 1000)

        seed = int(time.time() * 1000000) + os.getpid() * 1000 + k
        random.seed(seed)
        seeds.append(seed)
        partition_list = [0 for _ in range(n_nodes)]
        partition = list(asyn_lpa_communities(G))
        for comm_index, comm in enumerate(partition):
            for node in comm:
                partition_list[node] = comm_index + 1 # Index from 1
        partitions.append(partition_list)
    partitions = np.array(partitions)
    return partitions, seeds

In [ ]:
def calc_louvain(G):
    print('Calculating partitions')
    partitions = []
    seeds = []
    for k in trange(1000):
        seed = int(time.time() * 1000)
        random.seed(seed)
        seeds.append(seed)
        partition = community.best_partition(G)
        partition_list = []
        for _, comm_index in partition.items():
            partition_list.append(comm_index+1) # Louvain indexes from 0, so add 1
        partitions.append(partition_list)
    partitions = np.array(partitions)
    return partitions, seeds

In [3]:
def calc_partitions(G, algorithm, n_nodes):
    '''
    In all cases, the partitions returned at the end should be a 1000 x n numpy array,
    where 1000 is the number of runs of the community finding algorithm, and n is the
    number of nodes for the graph. Each entry in the array is the integer community label
    for that node during that run of the algorithm.

    For consistency, all outputs will index communities from 1.
    '''
    if algorithm == 'louvain':
        partitions, seeds = calc_louvain(G)
    elif algorithm == 'Infomap':
        partitions, seeds = calc_infomap(G, n_nodes)
    elif algorithm == 'lpa':
        partitions, seeds = calc_lpa(G, n_nodes)
    return partitions, seeds


In [4]:
mi = 2

mu = mi/10 
algorithm = 'Infomap'
graphs_folder = f'LFR_Graph_Data/mu_0_{mi}'
parts_folder = f'LFR_Graph_Data/Community_Data/{algorithm}/runs/'

if not os.path.exists('LFR_Graph_Data/Community_Data/'):
        os.mkdir('LFR_Graph_Data/Community_Data')

if not os.path.exists(f'LFR_Graph_Data/Community_Data/{algorithm}/'):       
        os.mkdir(f'LFR_Graph_Data/Community_Data/{algorithm}/')
        os.mkdir(f'LFR_Graph_Data/Community_Data/{algorithm}/runs')

for graph_loc in os.listdir(graphs_folder):
   
    graph_loc = os.path.join(graphs_folder, graph_loc)
    graph_contents = os.listdir(graph_loc)
    graph_yml = [x for x in graph_contents if x.endswith('yml')][0]
    graph_name = graph_yml.replace('.yml', '')
    run_path = f"LFR_Graph_Data/Community_Data/Infomap/runs/{graph_name}_mu_0_4_runs.npy"
    if os.path.exists(run_path):
          print(f'skipping graph: {graph_name} because it existis in: {run_path}')
          continue
    
    with open(os.path.join(graph_loc, graph_yml)) as f:
        graph_info = yaml.load(f, Loader=yaml.Loader)
    G = graph_info['G']
    n_nodes = graph_info['n']
    for node in G.nodes:
        del G.nodes[node]['community']
    partitions, seeds = calc_partitions(G, algorithm, n_nodes)
    graph_npy = graph_yml.split('.')[0] + '_runs.npy'
    path = os.path.join(parts_folder, graph_npy)
    print(path)
    print(graph_npy)
    np.save(path, partitions)
    graph_seed = graph_yml.split('.')[0] + '_seeds'
    seeds_path = os.path.join(parts_folder, graph_seed)
    with open(seeds_path, 'wb') as fp:
            pickle.dump(seeds, fp)

Calculating partitions


100%|██████████| 1000/1000 [01:34<00:00, 10.64it/s]


LFR_Graph_Data/Community_Data/Infomap/runs/graph_01_mu_0_2_runs.npy
graph_01_mu_0_2_runs.npy
Calculating partitions


100%|██████████| 1000/1000 [01:33<00:00, 10.72it/s]


LFR_Graph_Data/Community_Data/Infomap/runs/graph_010_mu_0_2_runs.npy
graph_010_mu_0_2_runs.npy
Calculating partitions


...

LFR_Graph_Data/Community_Data/Infomap/runs/graph_097_mu_0_2_runs.npy
graph_097_mu_0_2_runs.npy
Calculating partitions


100%|██████████| 1000/1000 [02:42<00:00,  6.15it/s]


LFR_Graph_Data/Community_Data/Infomap/runs/graph_098_mu_0_2_runs.npy
graph_098_mu_0_2_runs.npy
Calculating partitions


100%|██████████| 1000/1000 [02:23<00:00,  6.96it/s]


LFR_Graph_Data/Community_Data/Infomap/runs/graph_099_mu_0_2_runs.npy
graph_099_mu_0_2_runs.npy
